In [ ]:
import pandas as pd
import pandas as pd
import utils
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler, TargetEncoder
from sklearn.pipeline import Pipeline
import numpy as np

%load_ext autoreload
%autoreload 2

In [ ]:
data_folder = 'data/'
raw_train_df = pd.read_csv(data_folder + 'train.csv')
raw_test_df = pd.read_csv(data_folder + 'test.csv')
train_demo_df = pd.read_csv(data_folder + 'train_demographics.csv')
test_demo_df = pd.read_csv(data_folder + 'test_demographics.csv')
temp_calculations_folder_name = 'temp_calculations/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)

In [ ]:
feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns = ['acc_x', 'acc_y', 'acc_z']
rot_columns = ['rot_x', 'rot_y', 'rot_z']
temp_columns = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
tof_columns = raw_train_df.columns[raw_train_df.columns.str.startswith('tof')]
non_device_cols = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness', 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
sampling_rate = 100
exp_name = 'imu_only_simple'
exp_name = 'fft_v1_baseline' # Named for your experiment
sampling_rate = 100
eg_subject = 'SUBJ_040724'
single_sequence = 'SEQ_065189'
dc_offset_max = 10
phase_1_cols = ['acc_x', 'acc_y', 'acc_z', 'sequence_counter', 'phase', 'behavior', 'orientation', 'subject', 'gesture', 'sequence_id']
log_edges = np.logspace(np.log10(0.5), np.log10(100), num=15)

In [ ]:
train_df = raw_train_df.set_index('row_id')
train_df.head(1)

In [ ]:
single_subject_df = train_df.loc[train_df['subject'] == eg_subject, :]
single_gesture_df = single_subject_df.loc[single_subject_df['sequence_id'] == single_sequence]
single_gesture_df.head(1)

In [ ]:
train_df['sequence_id'].unique()

In [ ]:
sequences = ['SEQ_000034', 'SEQ_065478', 'SEQ_065470']

In [ ]:
multiple_gesture_df = train_df.loc[train_df['sequence_id'].isin(sequences)]

In [ ]:
band_edges = np.arange(0, 101, 10)
pipe = utils.ImuExtractor(imu_sensor_list=['acc_x', 'acc_y','acc_z'], 
                              sampling_rate=sampling_rate, 
                              domain='acceleration',
                              dc_offset=2.0,
                              band_edges=None)
pipe.transform(single_gesture_df)

In [ ]:
# Step 1: Process IMU values
fft_df = pipe.process_for_imu_values(single_gesture_df)
print("FFT DataFrame shape:", fft_df.shape)
print("FFT DataFrame index type:", type(fft_df.index))
print("FFT DataFrame columns:", fft_df.columns.tolist())
print("FFT DataFrame first few rows:\n", fft_df.head())

# Step 2: Extract features
features_df = utils.ImuExtractor.extract_features_from_imu(fft_df, band_edges=None)
print("Features DataFrame shape:", features_df.shape)
print("Features columns:", features_df.columns.tolist())
print("Features:\n", features_df)

In [ ]:
hm = pipe.process_for_imu_values(single_gesture_df)
hm = pipe.extract_features_from_imu(hm, None)
hm

In [ ]:
pipe.return_single_category_desc_record(single_gesture_df)

In [ ]:
rah = pipe.transform(single_gesture_df)
rah

In [ ]:
eg_subject = 'SUBJ_040724'


In [ ]:
category_row_df = single_gesture_df.groupby('sequence_id')[['orientation', 'subject', 'gesture']].agg(lambda x: x.unique()[0]).reset_index()
category_row_df

In [ ]:
temp_acc_time_df = single_gesture_df[acc_columns]
temp_acc_fft_df = temp_acc_time_df.apply(utils.convert_frame_to_fft, axis=0, args=(sampling_rate,))
temp_acc_fft_df.head(1)

In [ ]:
single_row_df = features_acc.stack()

single_row_df.index = [f"{axis}_{feat}" for axis, feat in single_row_df.index]
single_row_df = single_row_df.to_frame().T
single_row_df.index = [single_sequence]

In [ ]:
single_row_df

In [ ]:


single_row_df.loc[single_sequence, 'orientation'] = temp_orientation
single_row_df.loc[single_sequence, 'subject'] = temp_subject
single_row_df.loc[single_sequence, 'gesture'] = temp_gesture